[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/project_sessions_llama_index_query_engine.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">在 LlamaIndex 中使用会话</h1>

会话是一组追踪，表示用户与应用的一系列交互。

本教程将：
- 构建并追踪简单的 LlamaIndex 应用
- 使用会话组织追踪

ℹ️ 本 Notebook 需要 OpenAI API 密钥。

## 1. 安装依赖并导入库

In [ ]:
%pip install -Uq arize-phoenix "llama-index==0.14.0" openinference-instrumentation-llama-index gcsfs faker

In [ ]:
import json
import os
from getpass import getpass
from random import sample
from urllib.request import urlopen
from uuid import uuid4

from faker import Faker
from gcsfs import GCSFileSystem
from llama_index.core import (
    Settings,
    StorageContext,
    load_index_from_storage,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from openinference.instrumentation import using_session, using_user
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register
from tqdm import tqdm

import phoenix as px

## 2. 配置 OpenAI API 密钥

如果尚未通过环境变量设置 OpenAI API 密钥，请在此设置。

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
os.environ["OPENAI_API_KEY"] = openai_api_key

## 3. 配置默认项目并启动 Phoenix



🚨 Phoenix 通过环境变量配置。🚨

本教程会修改下方的 `PHOENIX_PROJECT_NAME` 环境变量，以更改追踪数据发送到的默认项目。

In [ ]:
os.environ["PHOENIX_PROJECT_NAME"] = "SESSIONS-DEMO"

通过 `LlamaIndexInstrumentor` 启用 Phoenix 追踪。Phoenix 使用 OpenInference 追踪：这是一种用于捕获和存储 LLM 应用追踪的开源标准，使应用能够无缝接入 Phoenix 等 LLM 可观测性解决方案。

In [ ]:
tracer_provider = register(endpoint="http://127.0.0.1:6006/v1/traces")
LlamaIndexInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

启动 Phoenix

In [ ]:
px.launch_app()

## 4. 构建 LlamaIndex 应用

本示例在预构建的 Arize 文档索引上使用 `RetrieverQueryEngine`，也可以换成自己的 LlamaIndex 应用。

从云存储下载预构建的 Arize 文档索引，并实例化存储上下文。

In [ ]:
file_system = GCSFileSystem(project="public-assets-275721")
persist_dir = "arize-phoenix-assets/datasets/unstructured/llm/llama-index/arize-docs/index/"
storage_context = StorageContext.from_defaults(fs=file_system, persist_dir=persist_dir)

现在可以实例化用于检索增强生成（RAG）的查询引擎。查询引擎是 LlamaIndex 中支持对数据提问的通用接口，接收自然语言查询并返回丰富的回答。它构建于检索器之上，也可以组合多个查询引擎实现更高级的能力。

In [ ]:
Settings.llm = OpenAI(model="gpt-4o-mini")
Settings.embed_model = OpenAIEmbedding()
index = load_index_from_storage(storage_context)
query_engine = index.as_query_engine()

# 5. 下载示例查询

In [ ]:
queries_url = "http://storage.googleapis.com/arize-phoenix-assets/datasets/unstructured/llm/context-retrieval/arize_docs_queries.jsonl"
with urlopen(queries_url) as response:
    queries = [json.loads(line)["query"] for line in response]

# 6. 按用户会话对查询分组

In [ ]:
session_id = str(uuid4())
session_user = Faker().user_name()

with using_session(session_id), using_user(session_user):
    for query in tqdm(sample(queries, 3)):
        query_engine.query(query)

<video controls src="https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/llama-index-knowledge-base-tutorial/project_sessions.mov" />